# EDA — StackOverflow NLP Query Engine

Exploratory pass over the loaded database: table sizes, tag distribution,
answer/acceptance patterns, time-to-first-answer. Run `src/build_db.py` and
`src/add_indexes.py` first — this notebook expects `stackoverflow.db` to
already exist at the project root.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path.cwd().parent / "stackoverflow.db"
conn = sqlite3.connect(DB_PATH)

## Table sizes

In [ ]:
tables = ["users", "tags", "posts_questions", "posts_answers", "post_tags", "comments", "votes"]
counts = {t: pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0] for t in tables}
pd.Series(counts, name="row_count").sort_values(ascending=False)

## Tag distribution

Question volume by tag — check whether the curated tag export (see
`docs/DATA_SCHEMA.md`) produced a reasonably balanced set or a long tail
dominated by one or two tags.

In [ ]:
tag_counts = pd.read_sql(
    """
    SELECT t.tag_name, COUNT(pt.post_id) AS question_count
    FROM tags t
    JOIN post_tags pt ON t.tag_id = pt.tag_id
    GROUP BY t.tag_id, t.tag_name
    ORDER BY question_count DESC
    LIMIT 30
    """,
    conn,
)
tag_counts.plot(kind="barh", x="tag_name", y="question_count", figsize=(8, 10), legend=False)
plt.gca().invert_yaxis()
plt.title("Top 30 tags by question volume")
plt.tight_layout()

## Answer acceptance rate over time

Fraction of answers marked accepted, by year — sanity-checks whether the
scoped export still shows realistic StackOverflow behavior.

In [ ]:
acceptance_by_year = pd.read_sql(
    """
    SELECT strftime('%Y', creation_date) AS year,
           AVG(is_accepted) AS acceptance_rate,
           COUNT(*) AS n_answers
    FROM posts_answers
    GROUP BY year
    ORDER BY year
    """,
    conn,
)
acceptance_by_year

## Time to first answer

Distribution of hours between question creation and its first answer
(same computation used in `src/queries.py::q6`, visualized here).

In [ ]:
deltas = pd.read_sql(
    """
    SELECT (julianday(MIN(a.creation_date)) - julianday(q.creation_date)) * 24 AS hours
    FROM posts_questions q
    JOIN posts_answers a ON a.parent_id = q.post_id
    GROUP BY q.post_id
    """,
    conn,
)
print("median hours to first answer:", deltas["hours"].median())
deltas[deltas["hours"] < 72]["hours"].plot(kind="hist", bins=50, title="Time to first answer (<72h)")
plt.xlabel("hours")

In [ ]:
conn.close()